# Building a RAG System with LangChain, LangGraph & OpenAI

This notebook demonstrates how to build a complete Retrieval-Augmented Generation (RAG) system using:
- **LangChain**: For document processing and LLM orchestration
- **LangGraph**: For workflow management
- **FAISS**: For in-memory vector storage and retrieval
- **OpenAI**: For embeddings and text generation

## Architecture Overview

1. **Document Loading**: Read documents from a folder
2. **Chunking**: Split documents into manageable pieces
3. **Vectorization**: Convert chunks to embeddings using OpenAI
4. **Storage**: Store vectors in-memory using FAISS
5. **Retrieval**: Find relevant chunks for user questions
6. **Prompt Creation**: Build system prompt with retrieved context
7. **Generation**: Generate answers using OpenAI
8. **Workflow**: Orchestrate everything with LangGraph


## Step 1: Setup and Dependencies

First, let's install and import all necessary packages.


In [ ]:
# Install required packages (run this once)
%pip install langchain langchain-openai langchain-community python-dotenv -q

# For document processing and vector storage
%pip install pypdf unstructured faiss-cpu -q


In [ ]:
import glob
import os
from typing import Any

from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema import Document

# LangChain imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [25]:
# Load environment variables
load_dotenv()

print("✅ All packages imported successfully!")

✅ All packages imported successfully!


In [26]:
# Configuration - Set your API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Set in .env file or replace with your key

# Validate API key
if not OPENAI_API_KEY:
    print(
        "⚠️ OPENAI_API_KEY not found. Please set it in your .env file or replace the variable above."
    )
    print("💡 You can get your API key from: https://platform.openai.com/api-keys")

# Configuration parameters
DOCUMENTS_FOLDER = "./documents"  # Folder containing your documents (supports nested folders)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

print("✅ Configuration loaded!")
print(f"📁 Documents folder: {DOCUMENTS_FOLDER}")
print(f"📏 Chunk size: {CHUNK_SIZE} with {CHUNK_OVERLAP} overlap")
print(f"🔤 Embedding model: {EMBEDDING_MODEL}")
print(f"🤖 LLM model: {LLM_MODEL}")


✅ Configuration loaded!
📁 Documents folder: ./documents
📏 Chunk size: 1000 with 200 overlap
🔤 Embedding model: text-embedding-3-small
🤖 LLM model: gpt-4o-mini


## Step 2: Document Loading

Load documents from a specified folder. This function supports various file types including PDF, TXT, and more.


In [70]:
# Document loading functions - refactored for better separation of concerns
SUPPORTED_FILE_TYPES = {
    "*.txt": TextLoader,
    "*.pdf": PyPDFLoader,
}


def _find_documents(folder_path: str, file_types: dict[str, type]) -> list[tuple[str, str, type]]:
    """
    Find all documents in a folder structure that match the file types in the dictionary.

    Args:
        folder_path: Path to the folder containing documents
        file_types: Dictionary of file types and their loaders

    Returns:
        List of tuples containing (file_path, company, loader_class)
    """
    if not os.path.exists(folder_path):
        print(f"⚠️ Folder {folder_path} does not exist!")
        return []

    found_files = []

    for pattern, loader_class in file_types.items():
        # Use recursive glob pattern to search in all subdirectories
        search_pattern = os.path.join(folder_path, "**", pattern)
        files = glob.glob(search_pattern, recursive=True)

        for file_path in files:
            # Skip index files (they're just metadata)
            if os.path.basename(file_path).startswith("_"):
                print(f"⏭️ Skipping index file: {file_path}")
                continue

            # Extract company/category from folder structure
            relative_path = os.path.relpath(file_path, folder_path)
            path_parts = relative_path.split(os.sep)
            company_category = path_parts[0] if len(path_parts) > 1 else "root"

            found_files.append((file_path, company_category, loader_class))

    return found_files


def _parse_single_document(
    file_path: str, company_category: str, loader_class: type, base_folder_path: str
) -> Document:
    """
    Parse a single document into a Document object with metadata.

    Args:
        file_path: Path to the document
        company_category: Category of the company
        loader_class: Loader class for the document
        base_folder_path: Base folder path for relative path calculation
    Returns:
        Document object
    """

    try:
        loader = loader_class(file_path)
        docs = loader.load()

        # Calculate relative path
        relative_path = os.path.relpath(file_path, base_folder_path)

        # Add metadata to each document
        for doc in docs:
            doc.metadata.update(
                {
                    "source_file": os.path.basename(file_path),
                    "file_type": os.path.splitext(file_path)[1],
                    "full_path": file_path,
                    "company": company_category,
                    "category": company_category,  # alias for company
                    "relative_path": relative_path,
                }
            )

        return docs[0]

    except Exception as e:
        print(f"❌ Error parsing {file_path}: {str(e)}")
        return None


def _parse_documents(
    found_files: list[tuple[str, str, type]],
    base_folder_path: str,
) -> list[Document]:
    """
    Parse found files into Document objects with metadata.

    Args:
        found_files: List of tuples containing (file_path, company, loader_class)
        base_folder_path: Base folder path for relative path calculation

    Returns:
        List of parsed Document objects
    """
    documents = []

    for file_path, company_category, loader_class in found_files:
        document = _parse_single_document(
            file_path, company_category, loader_class, base_folder_path
        )
        if document:
            documents.append(document)

    return documents


def load_documents_from_folder(folder_path: str, file_types: dict[str, type]) -> list[Document]:
    """
    Load documents from a folder supporting multiple file types and child folders.

    Args:
        folder_path: Path to the folder containing documents
        file_types: Dictionary of file types and their loaders

    Returns:
        List of LangChain Document objects
    """
    # Step 1: Find all documents to process
    found_files = _find_documents(folder_path, file_types)

    if not found_files:
        print(f"📚 No documents found in {folder_path}")
        return []

    # Step 2: Parse the found documents
    documents = _parse_documents(found_files, folder_path)

    # Step 3: Print summary statistics
    if documents:
        categories = {}
        for doc in documents:
            category = doc.metadata.get("category", "unknown")
            categories[category] = categories.get(category, 0) + 1

        print("\n📊 Documents loaded by category:")
        for category, count in sorted(categories.items()):
            print(f"  • {category}: {count} documents")

    print(f"\n📚 Total documents loaded: {len(documents)}")
    return documents


# Load documents (now supports child folders!)
documents = load_documents_from_folder(DOCUMENTS_FOLDER, file_types=SUPPORTED_FILE_TYPES)

# Show some example metadata
if documents:
    print("\n🔍 Example document metadata:")
    print("=" * 50)
    for i, doc in enumerate(documents[:2]):  # Show first 2 documents
        print(f"Document {i + 1}:")
        print(f"  • Source: {doc.metadata.get('source_file', 'Unknown')}")
        print(f"  • Company: {doc.metadata.get('company', 'Unknown')}")
        print(f"  • Path: {doc.metadata.get('relative_path', 'Unknown')}")
        print(f"  • Content preview: {doc.page_content[:100]}...")
        print("-" * 30)

documents[0]


⏭️ Skipping index file: ./documents/apple/_apple_index.txt
⏭️ Skipping index file: ./documents/amazon/_amazon_index.txt
⏭️ Skipping index file: ./documents/google/_google_index.txt
⏭️ Skipping index file: ./documents/microsoft/_microsoft_index.txt
⏭️ Skipping index file: ./documents/tesla/_tesla_index.txt

📊 Documents loaded by category:
  • amazon: 3 documents
  • apple: 3 documents
  • google: 3 documents
  • microsoft: 3 documents
  • tesla: 3 documents

📚 Total documents loaded: 15

🔍 Example document metadata:
Document 1:
  • Source: What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt
  • Company: apple
  • Path: apple/What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt
  • Content preview: COMPANY: APPLE
ARTICLE: 3/3
GENERATED: 2025-09-14 15:10:07
=========================================...
------------------------------
Document 2:
  • Source: Apple_Event_Live_Blog_Updates_on_iPhone_17_iPhone__20250914_151007.txt
  • Company: apple
  • Path: appl

Document(metadata={'source': './documents/apple/What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt', 'source_file': 'What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt', 'file_type': '.txt', 'full_path': './documents/apple/What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt', 'company': 'apple', 'category': 'apple', 'relative_path': 'apple/What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt'}, page_content="COMPANY: APPLE\nARTICLE: 3/3\nGENERATED: 2025-09-14 15:10:07\n================================================================================\n\nTITLE: What would actually make the Apple Watch better?\nSOURCE: The Verge\nAUTHOR: Victoria Song\nPUBLISHED: 2025-09-04T22:09:03Z\nURL: https://www.theverge.com/optimizer-newsletter/772003/apple-watch-iphone-17-optimizer-smartwatch-wearables\nDESCRIPTION: I've been thinking ahead to Apple's big event next Tuesday. I'll be sitting in Apple's Steve Jobs theater, ready to live

## Step 3: Document Chunking

Split documents into smaller, manageable chunks that can be effectively vectorized and retrieved.


In [ ]:
import json


def chunk_documents(
    documents: list[Document], chunk_size: int = 1000, chunk_overlap: int = 200
) -> list[Document]:
    """
    Split documents into smaller chunks for better retrieval.

    Args:
        documents: List of documents to chunk
        chunk_size: Maximum size of each chunk
        chunk_overlap: Overlap between consecutive chunks

    Returns:
        List of chunked documents
    """

    # Initialize the text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],  # Try to split on paragraphs, then lines, then words
    )

    # Split documents
    chunked_docs = text_splitter.split_documents(documents)

    # Add chunk information to metadata while preserving original metadata
    for i, chunk in enumerate(chunked_docs):
        # Preserve all original metadata and add chunk-specific information
        chunk.metadata.update(
            {
                "chunk_id": i,
                "chunk_size": len(chunk.page_content),
                "total_chunks": len(chunked_docs),
            }
        )

    print(f"📄 Split {len(documents)} documents into {len(chunked_docs)} chunks")

    # Show some statistics
    chunk_sizes = [len(chunk.page_content) for chunk in chunked_docs]
    print(
        f"📊 Chunk size stats: min={min(chunk_sizes)}, max={max(chunk_sizes)}, avg={sum(chunk_sizes) // len(chunk_sizes)}"
    )

    # Show company distribution in chunks
    company_chunks = {}
    for chunk in chunked_docs:
        company = chunk.metadata.get("company", "Unknown")
        company_chunks[company] = company_chunks.get(company, 0) + 1

    print(f"📊 Chunks by company: {dict(sorted(company_chunks.items()))}")

    return chunked_docs


# Chunk the loaded documents
chunks = chunk_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

# Display first chunk as example
print("\n🔍 Example chunk:")
print("=" * 50)
print(f"Content: {chunks[0].page_content[:200]}...")
print(f"Company: {chunks[0].metadata.get('company', 'Unknown')}")
print(f"Source File: {chunks[0].metadata.get('source_file', 'Unknown')}")
print(f"Chunk ID: {chunks[0].metadata.get('chunk_id', 'Unknown')}")

print(f"Full Metadata: {json.dumps(chunks[0].metadata, indent=4)}")
print("=" * 50)


📄 Split 15 documents into 80 chunks
📊 Chunk size stats: min=145, max=997, avg=794
📊 Chunks by company: {'amazon': 9, 'apple': 17, 'google': 20, 'microsoft': 20, 'tesla': 14}

🔍 Example chunk:
Content: COMPANY: APPLE
ARTICLE: 3/3
GENERATED: 2025-09-14 15:10:07

TITLE: What would actually make the Apple Watch better?
SOU...
Company: apple
Source File: What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt
Chunk ID: 0
Full Metadata: {
    "source": "./documents/apple/What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt",
    "source_file": "What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt",
    "file_type": ".txt",
    "full_path": "./documents/apple/What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt",
    "company": "apple",
    "category": "apple",
    "relative_path": "apple/What_would_actually_make_the_Apple_Watch_better_20250914_151007.txt",
    "chunk_id": 0,
    "chunk_size": 852,
    "total_chunks": 80
}


## Step 4: Vectorization with OpenAI Embeddings

Convert text chunks into vector embeddings using OpenAI's embedding models.


In [34]:
# Initialize OpenAI embeddings
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, openai_api_key=OPENAI_API_KEY)

# Test embeddings with a sample text
sample_text = "Machine learning is a subset of artificial intelligence."
sample_embedding = embeddings.embed_query(sample_text)

print("✅ Embeddings initialized!")
print(f"📊 Embedding model: {EMBEDDING_MODEL}")
print(f"📏 Embedding dimension: {len(sample_embedding)}")
print(f"🔍 Sample embedding (first 5 values): {sample_embedding[:5]}")


# Function to create embeddings for all chunks
def create_embeddings_for_chunks(chunks: list[Document], embeddings_model) -> list[list[float]]:
    """
    Create embeddings for all document chunks.

    Args:
        chunks: List of document chunks
        embeddings_model: OpenAI embeddings model

    Returns:
        List of embedding vectors
    """
    print(f"🔄 Creating embeddings for {len(chunks)} chunks...")

    # Extract text content from chunks
    texts = [chunk.page_content for chunk in chunks]

    # Create embeddings in batches for efficiency
    embeddings_list = embeddings_model.embed_documents(texts)

    print(f"✅ Created {len(embeddings_list)} embeddings")
    return embeddings_list


# Create embeddings for our chunks
chunk_embeddings = create_embeddings_for_chunks(chunks, embeddings)


✅ Embeddings initialized!
📊 Embedding model: text-embedding-3-small
📏 Embedding dimension: 1536
🔍 Sample embedding (first 5 values): [-0.021235132589936256, -0.05318146198987961, -0.01300511509180069, -0.028837835416197777, 0.055503468960523605]
🔄 Creating embeddings for 80 chunks...
✅ Created 80 embeddings


## Step 5: In-Memory Vector Storage with FAISS

Set up FAISS in-memory vector database and store the vectorized chunks for efficient similarity search.


In [37]:
def create_faiss_vector_store(chunks: list[Document], embeddings_model) -> FAISS:
    """
    Create FAISS vector store from document chunks.

    Args:
        chunks: List of document chunks
        embeddings_model: OpenAI embeddings model

    Returns:
        FAISS vector store
    """
    print(f"🔄 Creating FAISS vector store with {len(chunks)} documents...")

    # Extract texts and metadatas
    texts = [chunk.page_content for chunk in chunks]
    metadatas = [chunk.metadata for chunk in chunks]

    # Create FAISS vector store directly from texts
    # This will automatically generate embeddings for each text
    vector_store = FAISS.from_texts(texts=texts, embedding=embeddings_model, metadatas=metadatas)

    print("✅ Successfully created FAISS vector store!")
    print("📊 Vector store stats:")
    print(f"  • Total vectors: {vector_store.index.ntotal}")
    print(f"  • Vector dimension: {vector_store.index.d}")

    return vector_store


# Create FAISS vector store
vector_store = create_faiss_vector_store(chunks, embeddings)


🔄 Creating FAISS vector store with 80 documents...
✅ Successfully created FAISS vector store!
📊 Vector store stats:
  • Total vectors: 80
  • Vector dimension: 1536


In [ ]:
# Test retrieval
test_query = "What is the latest news about Amazon?"

# Test retrieval with scores
print(f"\n{'_' * 50}\n🔍 Testing retrieval with similarity scores...")
similar_docs_with_scores = vector_store.similarity_search_with_score(test_query, k=2)

print(f"Query: {test_query}")
for i, (doc, score) in enumerate(similar_docs_with_scores):
    print(f"\n--- Found Document {i + 1} (Score: {score:.4f}) ---")
    print(f"Content: {doc.page_content[:150]}...")
    print(f"Source: {doc.metadata.get('source_file', 'Unknown')}")



__________________________________________________
🔍 Testing retrieval with similarity scores...
Query: What is the latest news about Amazon?

--- Found Document 1 (Score: 0.9529) ---
Content: COMPANY: AMAZON
ARTICLE: 1/3
GENERATED: 2025-09-14 15:10:26

TITLE: A...
Source: Amazons_next_tablet_might_run_Android_20250914_151026.txt

--- Found Document 2 (Score: 0.9836) ---
Content: is a news writer who covers the streaming wars, consumer tech, crypto, social media, and much more. Previously, she was a writer and editor at MUO.

P...
Source: Amazons_next_tablet_might_run_Android_20250914_151026.txt


## Step 6: Retrieval System

Build a sophisticated retrieval system that finds the most relevant document chunks for a given query.


In [59]:
class RAGRetriever:
    """
    Advanced retrieval system for RAG pipeline.
    """

    def __init__(self, vector_store: FAISS, top_k: int = 5):
        self.vector_store = vector_store
        self.top_k = top_k

    def retrieve_documents(
        self,
        query: str,
        top_k: int = None,
        add_scores: bool = False,
    ) -> list[Document] | list[tuple]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: User query
            top_k: Number of documents to retrieve
            add_scores: If True, return (document, score) tuples; if False, return just documents

        Returns:
            List of relevant documents or list of (document, score) tuples
        """
        k = top_k or self.top_k

        if add_scores:
            # Perform similarity search with scores
            docs_with_scores = self.vector_store.similarity_search_with_score(query, k=k)
            return docs_with_scores

        # Perform similarity search without scores
        docs = self.vector_store.similarity_search(query, k=k)
        return docs

    def retrieve_with_scores(self, query: str, top_k: int = None) -> list[tuple]:
        """
        Retrieve documents with similarity scores.

        This method is kept for backward compatibility.

        Args:
            query: User query
            top_k: Number of documents to retrieve

        Returns:
            List of (document, score) tuples
        """
        return self.retrieve_documents(query, top_k, add_scores=True)

    def get_context_string(self, query: str, top_k: int = None) -> str:
        """
        Get formatted context string from retrieved documents.

        Args:
            query: User query
            top_k: Number of documents to retrieve

        Returns:
            Formatted context string
        """
        docs = self.retrieve_documents(query, top_k, add_scores=False)

        context_parts = []
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source_file", "Unknown")
            company = doc.metadata.get("company", "")
            company_prefix = f"({company}) " if company and company != "root" else ""
            content = doc.page_content.strip()
            context_parts.append(f"[Source {i}: {company_prefix}{source}]\\n{content}")

        return "\n\n".join(context_parts)


# Initialize retriever
retriever = RAGRetriever(vector_store, top_k=3)

# Test the retrieval system
print("🔍 Testing Retrieval System")
print("=" * 50)

test_queries = [
    "What are the latest news about Amazon?",
    "Tell me more about iphone 17",
    "I want to learn Machine Learning",
]

for query in test_queries:
    print(f"\n🤔 Query: {query}")

    # Get documents with scores
    docs_with_scores = retriever.retrieve_with_scores(query, top_k=2)

    print(f"📄 Retrieved {len(docs_with_scores)} documents:")
    for i, (doc, score) in enumerate(docs_with_scores, 1):
        print(f"\n--- Document {i} (Score: {score:.4f}) ---")
        print(f"Source: {doc.metadata.get('source_file', 'Unknown')}")
        print(f"Content: {doc.page_content[:200]}...")

print("\n" + "=" * 50)


🔍 Testing Retrieval System

🤔 Query: What are the latest news about Amazon?
📄 Retrieved 2 documents:

--- Document 1 (Score: 0.9399) ---
Source: Amazons_next_tablet_might_run_Android_20250914_151026.txt
Content: COMPANY: AMAZON
ARTICLE: 1/3
GENERATED: 2025-09-14 15:10:26

TITLE: Amazon’s next tablet might run Android
SOURCE: The ...

--- Document 2 (Score: 0.9731) ---
Source: Amazons_next_tablet_might_run_Android_20250914_151026.txt
Content: is a news writer who covers the streaming wars, consumer tech, crypto, social media, and much more. Previously, she was a writer and editor at MUO.

Posts from this author will be added to your daily ...

🤔 Query: Tell me more about iphone 17
📄 Retrieved 2 documents:

--- Document 1 (Score: 0.8516) ---
Source: Apple_Event_Live_Blog_Updates_on_iPhone_17_iPhone__20250914_151007.txt
Content: COMPANY: APPLE
ARTICLE: 2/3
GENERATED: 2025-09-14 15:10:07

TITLE: Apple Event Live Blog: Updates on iPhone 17, iPhone ...

--- Document 2 (Score: 1.0672) ---
Sou

Typical Score Ranges:
* 0.0 = Perfect match (identical vectors)
* 0.0 - 1.0 = Very high similarity
* 1.0 - 2.0 = Good similarity
* 2.0+ = Lower similarity

## Step 7: System Prompt Generation

Create dynamic system prompts using the retrieved context to guide the LLM's response.


In [ ]:
SYSTEM_TEMPLATE = """
You are a helpful AI assistant that answers questions based on the provided context. 

Your task is to:
1. Analyze the provided context carefully
2. Answer the user's question using information from the context
3. Be accurate and cite your sources when possible
4. If the context doesn't contain enough information, say so clearly
5. Maintain a helpful and professional tone

Context Information:
{context}

Guidelines:
- Only use information from the provided context
- If you're unsure about something, acknowledge the uncertainty
- Provide specific examples from the context when relevant
- Keep your answers comprehensive but concise
"""

HUMAN_TEMPLATE = """
Question: {question}
Please provide a detailed answer based on the context above.
"""

class RAGPromptGenerator:
    """
    Generate system prompts for RAG responses.
    """

    def __init__(self):
        # Define the system prompt template
        self.system_template = SYSTEM_TEMPLATE
        self.human_template = HUMAN_TEMPLATE

        # Create the prompt template
        self.prompt_template = ChatPromptTemplate.from_messages(
            [("system", self.system_template), ("human", self.human_template)]
        )

    def generate_prompt(self, question: str, context: str) -> ChatPromptTemplate:
        """
        Generate a complete prompt with context and question.

        Args:
            question: User's question
            context: Retrieved context from documents

        Returns:
            Formatted prompt template
        """
        return self.prompt_template.partial(context=context)

    def format_context(self, documents: list[Document]) -> str:
        """
        Format retrieved documents into context string.

        Args:
            documents: Retrieved documents

        Returns:
            Formatted context string
        """
        context_parts = []
        for i, doc in enumerate(documents, 1):
            source = doc.metadata.get("source_file", "Unknown")
            company = doc.metadata.get("company", "")
            company_prefix = f"({company}) " if company and company != "root" else ""
            content = doc.page_content.strip()
            context_parts.append(f"[Source {i}: {company_prefix}{source}]\\n{content}")

        return "\\n\\n".join(context_parts)


# Initialize prompt generator
prompt_generator = RAGPromptGenerator()

# Test prompt generation
print("📝 Testing Prompt Generation")
print("=" * 50)

test_question = "How long will the latest apple watch battery last?"
retrieved_docs = retriever.retrieve_documents(test_question, top_k=3)
context = prompt_generator.format_context(retrieved_docs)

# Generate the prompt
prompt = prompt_generator.generate_prompt(test_question, context)

# Display the formatted prompt
formatted_messages = prompt.format_messages(question=test_question)

print("🤖 Generated System Prompt:")
print("-" * 30)
for message in formatted_messages:
    print(f"**{message.type.upper()}:**")
    print(message.content)
    print("-" * 30)

print("✅ Prompt generation complete!")


📝 Testing Prompt Generation
🤖 Generated System Prompt:
------------------------------
**SYSTEM:**

You are a helpful AI assistant that answers questions based on the provided context. 

Your task is to:
1. Analyze the provided context carefully
2. Answer the user's question using information from the context
3. Be accurate and cite your sources when possible
4. If the context doesn't contain enough information, say so clearly
5. Maintain a helpful and professional tone

Context Information:
[Source 1: (apple) Apple_has_announced_the_Apple_Watch_Series_11_20250914_151007.txt]\nPrevious Next







1 / 5 The color options for the new Apple Watch Series 11. Screenshot: Apple

Apple says the Series 11 will get “up to 24 hours” of battery life. The aluminum version will come in jet black, space gray, rose gold, and silver; the polished titanium one will come in natural, gold, and slate. The watch also comes with Ion-X glass, which Apple says has a ceramic coating bonded at the atomic level,

## Step 8: Answer Generation

Generate comprehensive answers using OpenAI's language model with the retrieved context.


In [64]:
# Simplified RAGAnswerGenerator with only one method that always includes scores
class RAGAnswerGenerator:
    """
    Generate answers using retrieved context and LLM.
    Always includes similarity scores for transparency.
    """

    max_source_preview_length = 70

    def __init__(self, llm, retriever: RAGRetriever, prompt_generator: RAGPromptGenerator):
        self.llm = llm
        self.retriever = retriever
        self.prompt_generator = prompt_generator

    def generate_answer(self, question: str, top_k: int = 3) -> dict[str, Any]:
        """
        Generate answer with similarity scores for retrieved documents.

        Args:
            question: User's question
            top_k: Number of documents to retrieve

        Returns:
            Dictionary containing answer, sources with scores, and metadata
        """
        # Retrieve documents with scores
        docs_with_scores = self.retriever.retrieve_documents(question, top_k=top_k, add_scores=True)

        # Extract just documents for context generation
        retrieved_docs = [doc for doc, score in docs_with_scores]

        # Format context and generate answer
        context = self.prompt_generator.format_context(retrieved_docs)
        prompt = self.prompt_generator.generate_prompt(question, context)
        formatted_messages = prompt.format_messages(question=question)
        response = self.llm.invoke(formatted_messages)

        # Prepare result with scores
        result = {
            "question": question,
            "answer": response.content,
            "sources": [
                {
                    "file": doc.metadata.get("source_file", "Unknown"),
                    "company": doc.metadata.get("company", "Unknown"),
                    "content": doc.page_content[: self.max_source_preview_length] + "..."
                    if len(doc.page_content) > self.max_source_preview_length
                    else doc.page_content,
                    "similarity_score": float(score),
                }
                for doc, score in docs_with_scores
            ],
            "num_sources": len(retrieved_docs),
            "model_used": self.llm.model_name,
        }

        return result


# Initialize OpenAI Chat model
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0.1,  # Low temperature for more consistent, factual responses
    openai_api_key=OPENAI_API_KEY,
)

# Initialize simplified answer generator
answer_generator = RAGAnswerGenerator(llm, retriever, prompt_generator)


In [66]:
# Test the answer generator
print("🤖 Testing Simplified Answer Generation")
print("=" * 60)

test_questions = [
    "What are the latest news about Apple?",
    "Tell me about recent developments at Tesla",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n🤔 Question {i}: {question}")
    print("-" * 60)

    # Generate answer (always with scores now)
    result = answer_generator.generate_answer(question, top_k=2)

    print("🤖 **Answer:**")
    print(result["answer"])

    print(f"\n📚 **Sources Used ({result['num_sources']}):**")
    for j, source in enumerate(result["sources"], 1):
        company_info = (
            f" ({source['company']})"
            if source.get("company") and source["company"] != "Unknown"
            else ""
        )
        print(f"  {j}. {source['file']}{company_info} (Score: {source['similarity_score']:.4f})")
        print(f"     Content: {source['content']}")

    print(f"\n🔧 Model: {result['model_used']}")
    print("=" * 60)

print("✅ Simplified answer generation testing complete!")


🤖 Testing Simplified Answer Generation

🤔 Question 1: What are the latest news about Apple?
------------------------------------------------------------
🤖 **Answer:**
The latest news about Apple includes the announcement of several new products during their recent event. Key highlights are:

1. **iPhone 17 and iPhone Air**: Apple has launched the iPhone 17 and iPhone Air, although specific details about their features were not provided in the context.

2. **Apple Watch Series 11**: The Apple Watch Series 11 has been introduced as the slimmest model to date. It features 5G cellular connectivity, which is a first for the Apple Watch, and includes a redesigned cellular antenna for improved coverage in areas with weak signals. Additionally, the Series 11 will have live translation capabilities, similar to the upcoming AirPods Pro 3.

3. **AirPods Pro 3**: While specific details about the AirPods Pro 3 were not elaborated upon, it is mentioned that they will also include live translation ca

In [69]:
NO_RAG_SYSTEM_PROMPT = """
You are a helpful AI assistant that answers questions based on your training data knowledge. 

Your task is to:
1. Provide accurate information based on your knowledge
2. Be honest about the limitations of your knowledge
3. Acknowledge when information might be outdated
4. Maintain a helpful and professional tone
5. If you're unsure about recent developments, clearly state this

Please provide comprehensive answers while being transparent about the source of your information.
"""
for question in test_questions:
    no_rag_human_prompt = f"""Question: {question}
Please provide a detailed answer based on your knowledge.
"""
    no_rag_prompt = ChatPromptTemplate.from_messages([("system", NO_RAG_SYSTEM_PROMPT), ("human", no_rag_human_prompt)])
    no_rag_messages = no_rag_prompt.format_messages(question=question)
    no_rag_response = llm.invoke(no_rag_messages)
    print(f"{'-' * 40}\n🤖 **Answer without RAG (Direct LLM):**\n{no_rag_response.content}")


----------------------------------------
🤖 **Answer without RAG (Direct LLM):**
As of my last knowledge update in October 2023, I cannot provide real-time news or updates. However, I can summarize some of the key developments and trends related to Apple up to that point.

1. **Product Launches**: In September 2023, Apple typically holds its annual event where it announces new products. In 2023, Apple introduced the iPhone 15 series, which included improvements in camera technology, battery life, and performance. The iPhone 15 Pro models featured a titanium frame and USB-C charging, marking a significant change from previous models.

2. **Software Updates**: Apple released iOS 17 in September 2023, which included new features such as enhanced messaging capabilities, improved FaceTime options, and updates to privacy settings. Additionally, macOS Sonoma was launched, bringing new features to Mac users.

3. **Apple Vision Pro**: Apple continued to develop its mixed-reality headset, the App

## Next Steps & Production Considerations

### 💡 Why FAISS for This Tutorial?

**Advantages of In-Memory Storage:**
- ✅ **No API Keys Required**: No external service setup needed
- ✅ **Instant Setup**: Works immediately without configuration
- ✅ **Cost-Free**: Perfect for learning and development
- ✅ **Fast Performance**: In-memory operations are very fast
- ✅ **Offline Capable**: Works without internet connection
- ✅ **Persistence Option**: Can save/load indexes to/from disk

**When to Consider External Vector DBs:**
- 🔄 **Large Scale**: Millions of documents
- 🌐 **Multi-User**: Production applications
- 💾 **Persistence**: Long-term storage requirements
- 🔒 **Enterprise Features**: Advanced security, monitoring

### 🔧 Enhancements for Production

1. **Document Processing**
   - Add support for more file types (DOCX, HTML, CSV)
   - Implement document preprocessing (cleaning, normalization)
   - Add document metadata enrichment

2. **Chunking Strategies**
   - Experiment with different chunk sizes and overlaps
   - Implement semantic chunking based on content structure
   - Add chunk quality scoring

3. **Retrieval Improvements**
   - Implement hybrid search (vector + keyword)
   - Add query expansion and rewriting
   - Use re-ranking models for better relevance

4. **Vector Storage Scaling**
   - Move from FAISS to persistent vector databases (Pinecone, Weaviate, Qdrant)
   - Implement efficient batch updates
   - Add vector database monitoring and backup strategies

5. **LLM Integration**
   - Add response streaming for better UX
   - Implement response caching
   - Add support for multiple LLM providers

6. **Monitoring & Evaluation**
   - Add retrieval quality metrics
   - Implement answer quality scoring
   - Set up performance monitoring

### 🚀 Deployment Options

- **Local**: Run on local machine or server
- **Cloud**: Deploy on AWS, GCP, or Azure
- **Containerized**: Use Docker for consistent deployment
- **API**: Wrap in FastAPI or Flask for web service
- **Streamlit**: Create interactive web interface

### 📚 Additional Resources

- [LangChain Documentation](https://python.langchain.com/)
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [FAISS Documentation](https://faiss.ai/)
- [OpenAI API Documentation](https://platform.openai.com/docs)
- [Vector Database Comparison](https://github.com/langchain-ai/langchain/tree/master/libs/community/langchain_community/vectorstores)

---

**🎥 Perfect for your YouTube video! This notebook covers all the essential steps to build a production-ready RAG system.**


In [ ]:
# RAG vs Non-RAG Comparison
print("🆚 RAG vs Non-RAG Comparison")
print("=" * 80)
print("Let's compare answers with and without RAG to see the difference!")
print("=" * 80)

# Define no-RAG system and human prompts
no_rag_system_prompt = """You are a helpful AI assistant that answers questions based on your training data knowledge. 

Your task is to:
1. Provide accurate information based on your knowledge
2. Be honest about the limitations of your knowledge
3. Acknowledge when information might be outdated
4. Maintain a helpful and professional tone
5. If you're unsure about recent developments, clearly state this

Please provide comprehensive answers while being transparent about the source of your information."""

# Test questions for comparison
comparison_questions = [
    "What are the latest news about Apple?",
    "Tell me about recent developments at Tesla",
    "What's happening with Google's AI products?",
]

for i, question in enumerate(comparison_questions, 1):
    print(f"\n🔍 COMPARISON {i}/3")
    print(f"❓ Question: {question}")
    print("=" * 80)
    
    # 1. Generate answer WITHOUT RAG (direct LLM)
    print("🤖 **ANSWER WITHOUT RAG (Direct LLM):**")
    print("-" * 40)
    
    # Create no-RAG human prompt
    no_rag_human_prompt = f"""Question: {question}

Please provide a detailed answer based on your knowledge. If this involves recent developments or current events, please acknowledge any limitations in your knowledge."""

    # Create messages for the LLM
    messages = [
        {"role": "system", "content": no_rag_system_prompt},
        {"role": "user", "content": no_rag_human_prompt}
    ]
    
    direct_response = llm.invoke(messages)
    print(direct_response.content)
    print("📊 Sources: None (LLM knowledge only)")
    
    print("\n" + "-" * 80 + "\n")
    
    # 2. Generate answer WITH RAG
    print("🤖 **ANSWER WITH RAG (Context-Enhanced):**")
    print("-" * 40)
    
    # Use the existing answer generator (check which one is available)
    try:
        rag_result = answer_generator.generate_answer(question, top_k=2)
    except NameError:
        print("❌ answer_generator not found. Please run the previous cells first.")
        continue
    
    print(rag_result["answer"])
    
    print(f"\n📚 **RAG Sources Used ({rag_result['num_sources']}):**")
    for j, source in enumerate(rag_result["sources"], 1):
        company_info = (
            f" ({source['company']})"
            if source.get("company") and source["company"] != "Unknown"
            else ""
        )
        score_info = f" (Score: {source['similarity_score']:.4f})" if 'similarity_score' in source else ""
        print(f"  {j}. {source['file']}{company_info}{score_info}")
    
    print("\n" + "=" * 80)

print("\n🎯 **Key Differences:**")
print("✅ RAG Answers: Based on specific, recent documents with source attribution")
print("✅ RAG Answers: Include similarity scores for transparency")
print("✅ RAG Answers: Can provide company-specific, up-to-date information")
print("❌ Non-RAG Answers: Limited to LLM's training data (potentially outdated)")
print("❌ Non-RAG Answers: No source attribution or verification")
print("❌ Non-RAG Answers: May hallucinate or provide generic responses")

print("\n🚀 This demonstrates the power of RAG for domain-specific, up-to-date information retrieval!")
